In [1]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv

# ==========================================
# SETUP API
# ==========================================
load_dotenv()
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

output_file = "/workspaces/Arch_PC_Assistent/dataset/benchmark_questions.json"

def generate_benchmark_questions():
    print("Requesting 50 benchmark questions from DeepSeek API...")
    
    system_prompt = """You are an expert Linux System Administrator. Your task is to generate exactly 50 highly realistic, diverse questions that a user might ask an Arch Linux Assistant. 

DISTRIBUTION:
- 15 questions about Hyprland (configurations, monitors, waybar, portals, transparency, keybindings).
- 15 questions about Zsh (plugins, oh-my-zsh, scripting, path issues, aliases).
- 20 questions about general Arch Linux (pacman, yay, systemd, networking, grub).

DIFFICULTY:
Include a mix of simple "How do I..." questions and complex "I am getting this error..." troubleshooting scenarios.

OUTPUT FORMAT:
Return ONLY a valid JSON array of strings. Do not include markdown code blocks like ```json. Just the raw array.
Example: ["How do I update packages?", "Why is my Waybar not showing on the second monitor?"]"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "system", "content": system_prompt}],
            temperature=0.7, # Etwas Kreativität für abwechslungsreiche Fragen
        )
        
        raw_content = response.choices[0].message.content.strip()
        
        # Cleanup falls die API doch Markdown mitschickt
        if raw_content.startswith("```json"):
            raw_content = raw_content[7:]
        if raw_content.endswith("```"):
            raw_content = raw_content[:-3]
            
        questions = json.loads(raw_content)
        
        if len(questions) == 50:
            print("Successfully generated 50 questions!")
        else:
            print(f"Warning: Generated {len(questions)} questions instead of 50.")
            
        # Speichern der Fragen
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(questions, f, indent=4, ensure_ascii=False)
            
        print(f"Saved benchmark dataset to {output_file}")
        
    except Exception as e:
        print(f"Error during generation: {e}")

generate_benchmark_questions()

Requesting 50 benchmark questions from DeepSeek API...
Successfully generated 50 questions!
Saved benchmark dataset to /workspaces/Arch_PC_Assistent/dataset/benchmark_questions.json
